In [ ]:
%pip install -q grad-cam transformers

In [ ]:
import json, zipfile, requests
from pathlib import Path

import cv2
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

import torch
import torch.nn as nn
import torch.nn.functional as F
from transformers import SegformerForSemanticSegmentation, SegformerConfig
from pytorch_grad_cam import SegEigenCAM
from pytorch_grad_cam.utils.model_targets import SemanticSegmentationTarget
from pytorch_grad_cam.utils.image import show_cam_on_image

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DEVICE

In [ ]:
REPO = "williamtheodoruswijaya/PDIS-Net"
REPO_DIR = Path("/content/PDIS-Net")
DATA_DIR = Path("/content/flopwd_data")

if not REPO_DIR.exists():
    !GIT_LFS_SKIP_SMUDGE=1 git clone https://github.com/{REPO} {REPO_DIR}

%cd {REPO_DIR}

ROOT = REPO_DIR
MODEL_DIR = ROOT / "model"
XAI_OUT = ROOT / "xai" / "out"
XAI_OUT.mkdir(parents=True, exist_ok=True)

DATASET_ID = "114xnPVZtthHi4Bejzho33vGJSqb4-F8g"

if not DATA_DIR.exists():
    !gdown {DATASET_ID} -O /content/dataset.zip
    with zipfile.ZipFile("/content/dataset.zip") as z:
        z.extractall("/content/flopwd_data")

DATA = DATA_DIR / "Dal Lake Floating Plastic Waste Detection Dataset (FloPWD 2025)"
assert DATA.exists(), f"Dataset not found at {DATA}"

In [ ]:
RUN = "02-segformer-mitb3-imagenet"
VERSION = "v4-final"

PTH = MODEL_DIR / RUN / VERSION / "best.pth"
METRICS = MODEL_DIR / RUN / VERSION / "test_metrics.json"
SPLIT_TEST = ROOT / "results" / RUN / VERSION / "split_test.csv"

SAMPLES = ["img1484.jpg", "img324.jpg", "img2.jpg", "img563.jpg"]
MEAN = np.array([0.485, 0.456, 0.406], dtype=np.float32)
STD = np.array([0.229, 0.224, 0.225], dtype=np.float32)

In [ ]:
if PTH.stat().st_size < 1e6:
    url = f"https://media.githubusercontent.com/media/{REPO}/main/model/{RUN}/{VERSION}/best.pth"
    r = requests.get(url, stream=True)
    r.raise_for_status()
    with open(PTH, "wb") as f:
        for chunk in r.iter_content(8192):
            f.write(chunk)

print(f"{PTH.stat().st_size/1e6:.1f} MB : {PTH}")

In [ ]:
labels_df = pd.read_csv(DATA / "Image_labels_Binary Classification Task.csv").rename(columns={
    "image name": "image_name",
    "Presence of plastic waste?": "presence"
})
reg_df = pd.read_csv(DATA / "Mask_foreground_percentages_Regression Task.csv").rename(columns={
    "image name": "image_name",
    "plastic waste accumulation (in percentage)": "fg_pct"
})

df = labels_df.merge(reg_df, on="image_name", how="inner")
df["image_path"] = df["image_name"].apply(lambda x: str(DATA / "Raw_Images" / x))
df["mask_path"] = df["image_name"].str.replace(".jpg", "_mask.png", regex=False).apply(lambda x: str(DATA / "Segmentation_Masks" / x))
df["has_plastic"] = (df["presence"].astype(str).str.lower() == "yes").astype(int)

test_df = df[df.image_name.isin(pd.read_csv(SPLIT_TEST).image_name)].reset_index(drop=True)
len(test_df)

In [ ]:
class Logits(nn.Module):
    def __init__(self, model):
        super().__init__()
        self.model = model

    def forward(self, x):
        out = self.model(x)
        if hasattr(out, "logits"):
            return F.interpolate(out.logits, size=x.shape[-2:], mode="bilinear", align_corners=False)
        return out


class SegEigenCAMLite(SegEigenCAM):
    def get_cam_image(self, input_tensor, target_layer, targets, activations, grads, eigen_smooth):
        a = (np.abs(grads) * activations)[0]
        X = a.reshape(a.shape[0], -1).T
        X = X - X.mean(axis=0)
        _, _, VT = np.linalg.svd(X, full_matrices=False)
        p = (X @ VT[0]).reshape(a.shape[1:])
        if abs(p.min()) > abs(p.max()):
            p = -p
        return p[None]


def ious(p, t, eps=1e-7):
    tp = (p & t).sum()
    fp = (p & (1 - t)).sum()
    fn = ((1 - p) & t).sum()
    tn = ((1 - p) & (1 - t)).sum()
    return (tp + eps) / (tp + fp + fn + eps), (tn + eps) / (tn + fp + fn + eps)


def load_image(r, S):
    img = cv2.cvtColor(cv2.imread(r["image_path"]), cv2.COLOR_BGR2RGB)
    img = cv2.resize(img, (S, S))
    gt = (cv2.imread(r["mask_path"], cv2.IMREAD_GRAYSCALE) > 127).astype(np.uint8)
    gt = cv2.resize(gt, (S, S), interpolation=cv2.INTER_NEAREST)
    x = torch.from_numpy(((img / 255.0 - MEAN) / STD).transpose(2, 0, 1)).float()[None].to(DEVICE)
    return img, gt, x


def predict(model, x, threshold):
    with torch.no_grad():
        return (torch.softmax(model(x), 1)[0, 1] > threshold).cpu().numpy().astype(np.uint8)


def explain(cam, x, mask):
    return cam(x, [SemanticSegmentationTarget(1, mask.astype(np.float32))])[0]


def show_grid(names, title, df, model, cam, S, threshold, out_dir):
    valid = [n for n in names if n in df.image_name.values]
    if not valid:
        return
    fig, ax = plt.subplots(len(valid), 5, figsize=(20, 4 * len(valid)))
    if len(valid) == 1:
        ax = ax[None]
    for i, n in enumerate(valid):
        img, gt, x = load_image(df[df.image_name == n].iloc[0], S)
        pr = predict(model, x, threshold)
        panels = [
            (img, n),
            (gt, f"truth {gt.mean()*100:.2f}%"),
            (pr, f"pred {pr.mean()*100:.2f}%"),
            (show_cam_on_image(img / 255.0, explain(cam, x, pr), use_rgb=True), "why predicted"),
            (show_cam_on_image(img / 255.0, explain(cam, x, gt), use_rgb=True), "what in truth"),
        ]
        for a, (im, t) in zip(ax[i], panels):
            a.imshow(im, cmap="gray")
            a.set_title(t, fontsize=9)
            a.axis("off")
    fig.suptitle(title, fontsize=11)
    plt.tight_layout()
    plt.savefig(out_dir / f"{title}.png", dpi=90, bbox_inches="tight")
    plt.show()
    plt.close()


def faithfulness(per_img, df, model, cam, S, threshold):
    has = per_img[(per_img.gt_pct > 0) & (per_img.pred_pct > 0)]
    kept_hot, kept_cold = [], []
    for n in has.image_name:
        img, gt, x = load_image(df[df.image_name == n].iloc[0], S)
        pr = predict(model, x, threshold)
        c = explain(cam, x, pr)
        idx = np.flatnonzero(pr)
        if len(idx) < 2:
            continue
        order = idx[np.argsort(c.flat[idx])]
        k = len(idx) // 2
        for chosen, bucket in [(order[len(order)-k:], kept_hot), (order[:k], kept_cold)]:
            x2 = x.clone().flatten(2)
            x2[..., chosen] = 0
            bucket.append(float(predict(model, x2.view_as(x), threshold)[pr == 1].mean()))
    return kept_hot, kept_cold

In [ ]:
metrics = json.load(open(METRICS))
CFG = metrics["cfg"]
S = CFG["img_size"]
thr = CFG["threshold"]

net = SegformerForSemanticSegmentation(SegformerConfig.from_pretrained("nvidia/" + CFG["backbone"].replace("_", "-"), num_labels=2))
net.load_state_dict(torch.load(PTH, map_location="cpu", weights_only=True))
model = Logits(net).to(DEVICE).eval()
cam = SegEigenCAMLite(model, [net.decode_head.batch_norm])

out_dir = XAI_OUT / RUN / VERSION
out_dir.mkdir(parents=True, exist_ok=True)

print(f"arch={CFG['arch']} | backbone={CFG['backbone']} | init={CFG['init']} | mIoU={round(metrics['miou'], 4)}")

In [ ]:
rows = []
for _, r in test_df.iterrows():
    img, gt, x = load_image(r, S)
    pr = predict(model, x, thr)
    fg, bg = ious(pr, gt)
    rows.append({"image_name": r["image_name"], "fg_iou": fg,
                 "miou": (fg+bg)/2, "gt_pct": gt.mean()*100, "pred_pct": pr.mean()*100})

per_img = pd.DataFrame(rows)
reproduced = round(per_img["miou"].mean(), 4)
per_img.to_csv(out_dir / "per_image.csv", index=False)
print(f"reproduced={reproduced} | original={round(metrics['miou'], 4)}")

In [ ]:
has = per_img[per_img.gt_pct > 0].sort_values("fg_iou")

show_grid(SAMPLES, f"{RUN}_{VERSION}_samples", df, model, cam, S, thr, out_dir)

In [ ]:
show_grid(list(has.image_name[:3]), f"{RUN}_{VERSION}_worst", df, model, cam, S, thr, out_dir)

In [ ]:
show_grid(list(has.image_name[-3:]), f"{RUN}_{VERSION}_best", df, model, cam, S, thr, out_dir)

In [ ]:
kept_hot, kept_cold = faithfulness(per_img, df, model, cam, S, thr)

summary = {
    "run": RUN, "version": VERSION, "arch": CFG["arch"], "backbone": CFG["backbone"], "init": CFG["init"],
    "test_miou": metrics["miou"], "reproduced_miou": reproduced,
    "n_faithfulness_images": len(kept_hot),
    "kept_after_hot_delete": float(np.mean(kept_hot)),
    "kept_after_cold_delete": float(np.mean(kept_cold)),
    "hot_hurts_more_in": int(sum(h < c for h, c in zip(kept_hot, kept_cold))),
}
json.dump(summary, open(out_dir / "xai_summary.json", "w"), indent=2)
summary